## Generating training set of xTB water trajectories

At the time of this project, July 2026, the HPC facilities are inaccessible. Therefore, the training set generated here is generated on my home laptop, limiting the trajectories to small systems.

In [2]:
import ase.units as units
from ase.build import molecule
from ase.calculators.tip3p import TIP3P
from ase.constraints import FixBondLengths
from ase.io.trajectory import Trajectory
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md import Langevin
from ase.io import read, write
from ase.visualize import view

import matplotlib.pyplot as plt 
from ase.visualize.plot import plot_atoms 

import numpy as np

# randomly shuffling frames
import random 

When training ML models, it's equally, if not more importnant, to sample a variety of different configurations. Here, a single water molecule will be stretched and bended to obtain a wide range of configurations.

## Creating water molecule

Systematically sampling configuration space.

In [5]:
# create water molecule 
water = molecule('H2O')

# setting up range of bond lengths 
bond_lengths = np.linspace(0.7, 1.2, 10)

# setting up range of bond angles
angles = np.linspace(102, 106, 10)


for theta in angles:
    # set H-O-H angle
    theta = np.deg2rad(theta)
    for length1 in bond_lengths:
        for length2 in bond_lengths:
            pos = [[0.0, 0.0, 0.0],
                    [0.0, 0.0, length1],
                    [length2 * np.sin(theta), 0.0, length2 * np.cos(theta)]]

            # create water Atoms object with positions 
            water = molecule('H2O', positions=pos)
            
            # write current Atoms object 
            with open('water_sampling_1.xyz', 'a') as f:
                write(f, water, format='xyz')

print('Done systematic sampling.')


Done systematic sampling.


In [6]:
configs = read('water_sampling_1.xyz', index=':')
print(f'There are: {len(configs)} frames')

There are: 1000 frames


## Performing Single Point Calculators for water configurations

In [7]:
from ase.calculators.tip3p import TIP3P 
from ase.calculators.cp2k import CP2K
import os 
from ase.md.nptberendsen import NPTBerendsen

# load all frames from .xyz file 
configs = read('water_sampling_1.xyz', index=':')
os.environ['OMP_NUM_THREADS'] = '8'

print(os.environ['OMP_NUM_THREADS'])

calc = CP2K(inp='''                                       
&FORCE_EVAL
  &DFT
    &QS
      METHOD xTB
      &XTB
        CHECK_ATOMIC_CHARGES F
        COULOMB_INTERACTION T
        DO_EWALD T
      &END XTB
    &END QS
    &SCF
      SCF_GUESS ATOMIC
      EPS_SCF 1.0E-6
      &OT
        MINIMIZER DIIS
        ENERGY_GAP 0.1
        PRECONDITIONER FULL_SINGLE_INVERSE
      &END OT
    &END SCF

    CHARGE 0 
    MULTIPLICITY 1
  &END DFT
  &SUBSYS
    &TOPOLOGY
        COORD_FILE_FORMAT XYZ
        COORD_FILE_NAME water_sampling_1.xyz
        
    &END TOPOLOGY
  &END SUBSYS
&END FORCE_EVAL
''')
# creating dataset fize in extended .xyz format
dataset = 'water_dataset.extxyz'

for i, atoms in enumerate(configs):
    # set cell size and pbc
    atoms.set_cell([10.0, 10.0, 10.0])
    atoms.set_pbc(True)
    atoms.center()
    
    # attach calculator 
    atoms.calc = calc 
    
    # calculate energy
    energy = atoms.get_potential_energy()
    # calculate forces 
    forces = atoms.get_forces()

    # write to .extendedxyz
    write(dataset, atoms, append=True)
    print(f'{i} out of 1000 frames done')

print('All single point calculations done')


8
0 out of 343 frames done
1 out of 343 frames done
2 out of 343 frames done
3 out of 343 frames done
4 out of 343 frames done
5 out of 343 frames done
6 out of 343 frames done
7 out of 343 frames done
8 out of 343 frames done
9 out of 343 frames done
10 out of 343 frames done
11 out of 343 frames done
12 out of 343 frames done
13 out of 343 frames done
14 out of 343 frames done
15 out of 343 frames done
16 out of 343 frames done
17 out of 343 frames done
18 out of 343 frames done
19 out of 343 frames done
20 out of 343 frames done
21 out of 343 frames done
22 out of 343 frames done
23 out of 343 frames done
24 out of 343 frames done
25 out of 343 frames done
26 out of 343 frames done
27 out of 343 frames done
28 out of 343 frames done
29 out of 343 frames done
30 out of 343 frames done
31 out of 343 frames done
32 out of 343 frames done
33 out of 343 frames done
34 out of 343 frames done
35 out of 343 frames done
36 out of 343 frames done
37 out of 343 frames done
38 out of 343 frames

## Randomising frames in extxyz

In [8]:
import random
# read in frames of extxyz 
frames = read('water_dataset.extxyz', format='extxyz', index=':')

randomised_frames = random.shuffle(frames)
randomised_dataset = 'shuffled_water_dataset.extxyz'

write(randomised_dataset, frames, format='extxyz')
